# Transfer Learning v4 — Largest Modern CNNs + Advanced Training

Takes the best-performing architectures from v3 and pushes further with:

| # | Change vs v3 | Reason |
|---|---|---|
| 1 | **ConvNeXt-Small** + **EfficientNetV2-M** | Larger capacity — ConvNeXt-S matches ViT-S; EV2-M has 54 M params. |
| 2 | **288 × 288 input** | EfficientNetV2-M was trained at 384px; higher res recovers that advantage. |
| 3 | **WeightedRandomSampler** | Up-samples images with rare labels (clock, desk, keychain) for more effective gradient updates per epoch. |
| 4 | **Deeper head** — `Linear → BN → GELU → Dropout → Linear` | More expressive classification head for the richer backbone features. |
| 5 | **ASL + label smoothing** ε=0.05 | Calibrates logits; makes per-class threshold tuning more effective. |
| 6 | **Cosine Annealing with Warm Restarts** (SGDR) | Escapes local minima during full fine-tune; often finds flatter minima. |
| 7 | **5-view TTA** at inference | Averages predictions over flip + multi-scale crops for ~+1 F1. |

## 1. Imports & Setup

In [ ]:
import sys
import copy
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from pathlib import Path
from collections import defaultdict
from torchvision import models as tv_models
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler

sys.path.append("../")
from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset,
    plot_per_class_examples, plot_multilabel_examples,
    run_baselines,
    save_checkpoint,
    plot_multi_arch_histories,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, plot_prediction_heatmap,
    show_saliency_examples, compute_multilabel_metrics,
    NUM_LABELS, METRIC_KEYS, NORM_MEAN, NORM_STD, TransformSubset,
)

SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Config

In [ ]:
BASE_DATA_DIR    = "../data/aggregated"
IMAGE_SIZE       = 288        # higher res for V2-M; ConvNeXt-S also benefits
BATCH_SIZE       = 24         # reduced for 288px
SPLIT            = [0.7, 0.15, 0.15]
USE_SUBSET       = False
SUBSET_FRACTION  = 0.1
CHECKPOINT_DIR   = Path("../checkpoints")

# Two-stage training
FREEZE_EPOCHS    = 3
UNFREEZE_EPOCHS  = 20
MAX_EPOCHS       = FREEZE_EPOCHS + UNFREEZE_EPOCHS
LR_HEAD          = 1e-3
LR_BACKBONE      = 5e-5       # slightly more conservative for larger backbones
WEIGHT_DECAY     = 1e-4
GRAD_CLIP        = 1.0
THRESHOLD        = 0.5
THRESHOLD_GRID   = np.arange(0.05, 0.96, 0.02)

print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")
print(f"Total epochs: {MAX_EPOCHS}  (freeze={FREEZE_EPOCHS}, unfreeze={UNFREEZE_EPOCHS})")

## 3. Asymmetric Loss with Label Smoothing

Label smoothing ε=0.05 softens the hard 0/1 targets before ASL focuses on
hard examples — this produces better-calibrated probabilities, which in turn
makes per-class threshold tuning more reliable.

In [ ]:
class AsymmetricLossWithSmoothing(nn.Module):
    """ASL (Ben-Baruch 2020) + label smoothing."""

    def __init__(self, gamma_neg: float = 4, gamma_pos: float = 1,
                 clip: float = 0.05, eps: float = 1e-8, label_smooth: float = 0.05):
        super().__init__()
        self.gamma_neg    = gamma_neg
        self.gamma_pos    = gamma_pos
        self.clip         = clip
        self.eps          = eps
        self.label_smooth = label_smooth

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        if self.label_smooth > 0:
            targets = targets * (1 - self.label_smooth) + 0.5 * self.label_smooth
        probs_pos = torch.sigmoid(logits)
        probs_neg = 1 - probs_pos
        if self.clip > 0:
            probs_neg = (probs_neg + self.clip).clamp(max=1.0)
        loss_pos = targets       * torch.log(probs_pos.clamp(min=self.eps))
        loss_neg = (1 - targets) * torch.log(probs_neg.clamp(min=self.eps))
        loss = loss_pos + loss_neg
        pt    = probs_pos * targets + probs_neg * (1 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
        return -(loss * (1 - pt).pow(gamma)).mean()

criterion = AsymmetricLossWithSmoothing(gamma_neg=4, gamma_pos=1, clip=0.05, label_smooth=0.05)

## 4. Data — Stratified Split + WeightedRandomSampler + RandAugment

`WeightedRandomSampler` assigns each training sample a weight equal to
the sum of inverse class frequencies across its positive labels. Images
with rare labels (clock, desk) are drawn more often each epoch.

In [ ]:
def stratified_split_multilabel(dataset, split, seed=42):
    combo_to_indices = defaultdict(list)
    for i in range(len(dataset)):
        _, target = dataset[i]
        combo_to_indices[tuple(target.int().tolist())].append(i)
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for indices in combo_to_indices.values():
        indices = np.array(indices)
        rng.shuffle(indices)
        n       = len(indices)
        n_val   = max(1 if n >= 3 else 0, round(split[1] * n))
        n_test  = max(1 if n >= 3 else 0, round(split[2] * n))
        n_train = max(0, n - n_val - n_test)
        train_idx.extend(indices[:n_train].tolist())
        val_idx.extend(indices[n_train : n_train + n_val].tolist())
        test_idx.extend(indices[n_train + n_val :].tolist())
    return Subset(dataset, train_idx), Subset(dataset, val_idx), Subset(dataset, test_idx)

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.75, 1.0)),
    transforms.RandomHorizontalFlip(),
    RandAugment(num_ops=2, magnitude=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    RandomErasing(p=0.3, scale=(0.02, 0.15), value=0.0),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = stratified_split_multilabel(full_dataset, SPLIT, SEED)

if USE_SUBSET:
    from utils import subsample_subset
    train_raw = subsample_subset(train_raw, SUBSET_FRACTION, SEED)
    val_raw   = subsample_subset(val_raw,   SUBSET_FRACTION, SEED + 1)
    test_raw  = subsample_subset(test_raw,  SUBSET_FRACTION, SEED + 2)

# Build weighted sampler from TRAIN labels only
train_targets = torch.stack([full_dataset[i][1] for i in train_raw.indices])
label_counts  = train_targets.sum(dim=0).clamp(min=1)
inv_freq      = 1.0 / label_counts
sample_weights = (train_targets * inv_freq).sum(dim=1)
sampler = WeightedRandomSampler(
    weights=sample_weights.double(),
    num_samples=len(train_raw),
    replacement=True,
)

train_ds = TransformSubset(train_raw, transform=train_transform)
val_ds   = TransformSubset(val_raw,   transform=eval_transform)
test_ds  = TransformSubset(test_raw,  transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Total: {len(full_dataset)}  |  Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")
print(f"Sample weight range: {sample_weights.min():.4f} .. {sample_weights.max():.4f}")
print(f"Label counts (train): {dict(zip(LABEL_ORDER, label_counts.int().tolist()))}")

## 5. Baselines

In [ ]:
run_baselines(train_loader, val_loader, test_loader, NUM_LABELS, DEVICE)

## 6. Models — ConvNeXt-Small & EfficientNetV2-M

### Head design
A deeper head — `Linear(in, 512) → BN → GELU → Dropout(0.3) → Linear(512, K)` —
gives the model more capacity to compose backbone features into multi-label decisions.

In [ ]:
class TransferModel(nn.Module):
    def __init__(self, backbone: nn.Module, head: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.head     = head

    def forward(self, x):
        return self.head(self.backbone(x))

    def set_backbone_trainable(self, flag: bool):
        for p in self.backbone.parameters():
            p.requires_grad = flag


def _deep_head(in_features: int, num_labels: int, hidden: int = 512,
               dropout: float = 0.3) -> nn.Sequential:
    """Linear → BN → GELU → Dropout → Linear."""
    head = nn.Sequential(
        nn.Linear(in_features, hidden),
        nn.BatchNorm1d(hidden),
        nn.GELU(),
        nn.Dropout(dropout),
        nn.Linear(hidden, num_labels),
    )
    for layer in head.modules():
        if isinstance(layer, nn.Linear):
            nn.init.normal_(layer.weight, 0, 0.01)
            nn.init.zeros_(layer.bias)
    return head


def build_model(arch: str, num_labels: int) -> TransferModel:
    if arch == "convnext_small":
        m        = tv_models.convnext_small(weights=tv_models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
        backbone = nn.Sequential(m.features, m.avgpool, nn.Flatten())
        head     = _deep_head(768, num_labels)  # ConvNeXt-S: 768-dim output

    elif arch == "efficientnet_v2_m":
        m        = tv_models.efficientnet_v2_m(weights=tv_models.EfficientNet_V2_M_Weights.IMAGENET1K_V1)
        backbone = nn.Sequential(m.features, m.avgpool, nn.Flatten())
        head     = _deep_head(1280, num_labels)

    elif arch == "convnext_base":
        # Optional: even larger. May be too slow without a V100/A100.
        m        = tv_models.convnext_base(weights=tv_models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
        backbone = nn.Sequential(m.features, m.avgpool, nn.Flatten())
        head     = _deep_head(1024, num_labels)

    else:
        raise ValueError(f"Unknown arch: {arch}")

    return TransferModel(backbone, head)


# convnext_base is optional — comment out if GPU memory is tight
ARCHS = ["convnext_small", "efficientnet_v2_m", "convnext_base"]

print(f"{'Arch':<22} {'Total params':>14}  {'Size (MB)':>10}")
print("-" * 52)
for arch in ARCHS:
    m     = build_model(arch, NUM_LABELS)
    total = sum(p.numel() for p in m.parameters())
    print(f"{arch:<22} {total:>14,}  {total*4/1024/1024:>10.2f}")

## 7. Training — Two-Stage + Cosine Annealing with Warm Restarts (SGDR)

After unfreezing the backbone, we use `CosineAnnealingWarmRestarts` with
`T_0=5` (restart every 5 epochs). Warm restarts can escape sharp minima
that plain cosine decay gets stuck in.

In [ ]:
def run_epoch(model, loader, optimizer=None, train=True):
    model.train(train)
    all_logits, all_probs, all_preds, all_labels = [], [], [], []
    total_loss = 0.0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP
                )
                optimizer.step()
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= THRESHOLD).float()
        total_loss  += loss.item() * images.size(0)
        all_logits.append(logits.detach().cpu())
        all_probs.append(probs.detach().cpu())
        all_preds.append(preds.detach().cpu())
        all_labels.append(labels.cpu())
    n = sum(l.size(0) for l in all_labels)
    metrics = compute_multilabel_metrics(
        torch.cat(all_labels), torch.cat(all_preds),
        probs=torch.cat(all_probs), logits=torch.cat(all_logits),
    )
    metrics["loss"] = total_loss / n
    return metrics


def train_two_stage(arch):
    model = build_model(arch, NUM_LABELS).to(DEVICE)

    # Stage A — head only
    model.set_backbone_trainable(False)
    optimizer = optim.AdamW(
        [{"params": list(model.head.parameters()), "lr": LR_HEAD}],
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FREEZE_EPOCHS)

    best_val_f1, best_state = -1.0, None
    history = {"train": {k: [] for k in METRIC_KEYS}, "val": {k: [] for k in METRIC_KEYS}}

    print(f"[Stage A] backbone frozen for {FREEZE_EPOCHS} warmup epochs")
    for epoch in range(1, MAX_EPOCHS + 1):
        if epoch == FREEZE_EPOCHS + 1:
            model.set_backbone_trainable(True)
            optimizer = optim.AdamW(
                [
                    {"params": list(model.head.parameters()),     "lr": LR_HEAD},
                    {"params": list(model.backbone.parameters()), "lr": LR_BACKBONE},
                ],
                weight_decay=WEIGHT_DECAY,
            )
            # SGDR: restart every 5 epochs, multiply period by 2 each restart
            scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
                optimizer, T_0=5, T_mult=2, eta_min=1e-6
            )
            print(f"[Stage B] backbone unfrozen — SGDR fine-tune for {UNFREEZE_EPOCHS} epochs")

        t0      = time.time()
        train_m = run_epoch(model, train_loader, optimizer, train=True)
        val_m   = run_epoch(model, val_loader, train=False)
        scheduler.step()

        for k in METRIC_KEYS:
            history["train"][k].append(train_m[k])
            history["val"][k].append(val_m[k])

        stage = "A" if epoch <= FREEZE_EPOCHS else "B"
        print(f"[{stage}] {epoch:02d}/{MAX_EPOCHS} | {time.time()-t0:4.1f}s | "
              f"train f1={train_m['f1_micro']:.4f} | val f1={val_m['f1_micro']:.4f}")
        if val_m["f1_micro"] > best_val_f1:
            best_val_f1 = val_m["f1_micro"]
            best_state  = copy.deepcopy(model.state_dict())
            print(f"       -> new best {best_val_f1:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    return model, best_val_f1, history

In [ ]:
all_histories = {}
best_models   = {}
best_val_f1s  = {}

for arch in ARCHS:
    print(f"\n{'='*60}\n  {arch}\n{'='*60}")
    m, best_f1, history = train_two_stage(arch)
    save_checkpoint(m.state_dict(), CHECKPOINT_DIR / f"tl_v4_{arch}.pth")
    all_histories[arch] = {k: {"train": history["train"][k], "val": history["val"][k]}
                           for k in METRIC_KEYS}
    best_models[arch]   = m
    best_val_f1s[arch]  = best_f1

print("\n=== Val F1 Summary ===")
for arch in ARCHS:
    print(f"  {arch:<22} {best_val_f1s[arch]:.4f}")

plot_multi_arch_histories(all_histories, experiment_name="Transfer Learning v4")

## 8. Per-Class Threshold Tuning

In [ ]:
def tune_thresholds(model, val_loader, grid):
    all_probs, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in val_loader:
            probs = torch.sigmoid(model(images.to(DEVICE)))
            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())
    probs  = torch.cat(all_probs)
    labels = torch.cat(all_labels)
    thresholds = []
    for c in range(labels.shape[1]):
        best_t, best_f1 = 0.5, -1.0
        for t in grid:
            preds = (probs[:, c] >= t).float()
            tp = ((preds == 1) & (labels[:, c] == 1)).sum().float()
            fp = ((preds == 1) & (labels[:, c] == 0)).sum().float()
            fn = ((preds == 0) & (labels[:, c] == 1)).sum().float()
            f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).item()
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds.append(float(best_t))
    return thresholds, probs, labels


per_class_thresholds = {}
print("Tuning per-class thresholds on val set...\n")
for arch, m_eval in best_models.items():
    thresholds, val_probs, val_labels = tune_thresholds(m_eval, val_loader, THRESHOLD_GRID)
    per_class_thresholds[arch] = thresholds
    thr_t     = torch.tensor(thresholds).unsqueeze(0)
    val_preds = (val_probs >= thr_t).float()
    m         = compute_multilabel_metrics(val_labels, val_preds, probs=val_probs)
    print(f"{arch:<22}  val F1@tuned: {m['f1_micro']:.4f}")
    for i, cls in enumerate(LABEL_ORDER):
        print(f"  {cls:<12}  thr={thresholds[i]:.2f}")
    print()

## 9. 5-View TTA Inference

Views: original @ size, hflip @ size, original @ size+64 → crop, hflip @ size+64 → crop,
original @ size-32 → pad. Probabilities are averaged before thresholding.

In [ ]:
@torch.no_grad()
def tta_probs(model, images, base_size=IMAGE_SIZE):
    """5-view TTA: {orig, hflip} × {base, +64→crop} + {orig, -32→pad}."""
    def _resize_crop_or_pad(batch, new_size, target_size):
        r = F.interpolate(batch, size=(new_size, new_size), mode="bilinear", align_corners=False)
        if new_size == target_size:
            return r
        if new_size > target_size:
            off = (new_size - target_size) // 2
            return r[:, :, off:off + target_size, off:off + target_size]
        pad = (target_size - new_size) // 2
        return F.pad(r, (pad, pad, pad, pad), value=0.0)

    orig  = images
    flip  = torch.flip(images, dims=[3])
    views = [
        orig,
        flip,
        _resize_crop_or_pad(orig, base_size + 64, base_size),
        _resize_crop_or_pad(flip, base_size + 64, base_size),
        _resize_crop_or_pad(orig, base_size - 32, base_size),
    ]
    probs = torch.stack([torch.sigmoid(model(v)) for v in views], dim=0)
    return probs.mean(dim=0)


def collect_tta_predictions(model, loader):
    all_probs, all_labels = [], []
    model.eval()
    for images, labels in loader:
        images = images.to(DEVICE)
        all_probs.append(tta_probs(model, images).cpu())
        all_labels.append(labels)
    return torch.cat(all_probs), torch.cat(all_labels)

## 10. Test Evaluation — Default / TTA / TTA+Tuned Thresholds

In [ ]:
print(f"\n{'Arch':<22} {'F1@0.5':>8} {'F1@TTA':>8} {'F1@TTA+thr':>12} {'Exact':>7} {'IoU':>7}")
print("-" * 74)

tta_results = {}
for arch, m_eval in best_models.items():
    thr_t = torch.tensor(per_class_thresholds[arch]).unsqueeze(0)

    # Collect plain probs (no TTA)
    all_l, all_pr, all_lg = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            logits = m_eval(images.to(DEVICE))
            all_l.append(labels.cpu())
            all_pr.append(torch.sigmoid(logits).cpu())
            all_lg.append(logits.cpu())
    labels_t  = torch.cat(all_l)
    probs_t   = torch.cat(all_pr)
    logits_t  = torch.cat(all_lg)

    # TTA probs
    tta_probs_t, _ = collect_tta_predictions(m_eval, test_loader)
    tta_results[arch] = (tta_probs_t, labels_t, thr_t)

    m_def      = compute_multilabel_metrics(labels_t, (probs_t >= 0.5).float(),
                                            probs=probs_t, logits=logits_t)
    m_tta      = compute_multilabel_metrics(labels_t, (tta_probs_t >= 0.5).float(),
                                            probs=tta_probs_t)
    m_tta_thr  = compute_multilabel_metrics(labels_t, (tta_probs_t >= thr_t).float(),
                                            probs=tta_probs_t)
    print(f"{arch:<22} {m_def['f1_micro']:>8.4f} {m_tta['f1_micro']:>8.4f} "
          f"{m_tta_thr['f1_micro']:>12.4f} {m_tta_thr['exact_match']:>7.4f} "
          f"{m_tta_thr['mean_iou']:>7.4f}")

## 11. Analyze Predictions (Best Architecture + TTA)

In [ ]:
BEST_ARCH = max(best_val_f1s, key=best_val_f1s.get)
model     = best_models[BEST_ARCH]
print(f"Best arch: {BEST_ARCH}  (val F1={best_val_f1s[BEST_ARCH]:.4f})")

tta_probs_t, labels_t, thr_t = tta_results[BEST_ARCH]
preds_t = (tta_probs_t >= thr_t).float()

# collect_test_predictions returns images (needed for display) — run without TTA
images_t, _, _, _ = collect_test_predictions(model, test_loader, DEVICE)
correct_idx, partial_idx, incorrect_idx = categorize_predictions(labels_t, preds_t)

show_prediction_examples(correct_idx,   images_t, labels_t, preds_t, "Fully Correct",     n=4)
show_prediction_examples(partial_idx,   images_t, labels_t, preds_t, "Partially Correct", n=4)
show_prediction_examples(incorrect_idx, images_t, labels_t, preds_t, "Fully Incorrect",   n=4)

In [ ]:
plot_per_class_metrics(labels_t, preds_t)
plot_confusion_matrices(labels_t, preds_t)
plot_prediction_heatmap(labels_t, preds_t)

## 12. Saliency Maps

In [ ]:
show_saliency_examples(correct_idx,   images_t, labels_t, preds_t, model, "Fully Correct",     n=5, device=DEVICE)
show_saliency_examples(partial_idx,   images_t, labels_t, preds_t, model, "Partially Correct", n=5, device=DEVICE)
show_saliency_examples(incorrect_idx, images_t, labels_t, preds_t, model, "Fully Incorrect",   n=5, device=DEVICE)